In [1]:
import pandas as pd
import kagglehub
import os

# 1. Configuração do MinIO usando o DNS interno do Docker
minio_storage_options = {
    "key": "minioadmin",
    "secret": "minioadmin123",
    "client_kwargs": {
        "endpoint_url": "http://minio:9000" # Acesso direto container-a-container
    }
}

# 2. Download do Dataset (o kagglehub gerencia o cache no container)
print("Baixando dataset do Kaggle...")
dataset_dir = kagglehub.dataset_download("dhivyeshrk/diseases-and-symptoms-dataset")

# 3. Montagem dinâmica do caminho e leitura
file_name = "Final_Augmented_dataset_Diseases_and_Symptoms.csv"
csv_path = os.path.join(dataset_dir, file_name)

print(f"Lendo arquivo de: {csv_path}")
df = pd.read_csv(csv_path)
display(df.head())

# 4. Salvando direto no bucket da Camada Bronze
path_bronze = "s3://bronze/dataset_doencas_sintomas.csv"
df.to_csv(path_bronze, index=False, storage_options=minio_storage_options)
print(f"✅ Artefato Bronze salvo com sucesso no MinIO: {path_bronze}")

Baixando dataset do Kaggle...


100%|██████████| 2.81M/2.81M [00:05<00:00, 516kB/s]

Extracting files...


Lendo arquivo de: /home/jovyan/.cache/kagglehub/datasets/dhivyeshrk/diseases-and-symptoms-dataset/versions/1/Final_Augmented_dataset_Diseases_and_Symptoms.csv


,diseases,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain,dizziness,insomnia,abnormal involuntary movements,chest tightness,...,stuttering or stammering,problems with orgasm,nose deformity,lump over jaw,sore in nose,hip weakness,back swelling,ankle stiffness or tightness,ankle weakness,neck weakness
0,panic disorder,1,0,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,panic disorder,0,0,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,panic disorder,1,1,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,panic disorder,1,0,0,1,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,panic disorder,1,1,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0


✅ Artefato Bronze salvo com sucesso no MinIO: s3://bronze/dataset_doencas_sintomas.csv


Agrupar os sintomas condizentes a cada doença

In [2]:
col_doenca = 'diseases'

sintomas_cols = df.columns.drop(col_doenca)

def extrair_sintomas_linha(row):
    return [col for col in sintomas_cols if row[col] == 1]

# cria lista de sintomas por linha
df['sintomas'] = df.apply(extrair_sintomas_linha, axis=1)

In [3]:
df.head()

,diseases,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain,dizziness,insomnia,abnormal involuntary movements,chest tightness,...,problems with orgasm,nose deformity,lump over jaw,sore in nose,hip weakness,back swelling,ankle stiffness or tightness,ankle weakness,neck weakness,sintomas
0,panic disorder,1,0,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,"[anxiety and nervousness, shortness of breath,..."
1,panic disorder,0,0,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,"[shortness of breath, depressive or psychotic ..."
2,panic disorder,1,1,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,"[anxiety and nervousness, depression, shortnes..."
3,panic disorder,1,0,0,1,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,"[anxiety and nervousness, depressive or psycho..."
4,panic disorder,1,1,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,"[anxiety and nervousness, depression, insomnia..."


Organizar para o silver um dataset mais bem estruturado

In [4]:
import re

# Remove colunas duplicadas do pandas (ex: regurgitation.1)
sintomas_cols_limpos = [col for col in sintomas_cols if not re.search(r'\.\d+$', col)]

def extrair_sintomas_linha_limpo(row):
    return [col for col in sintomas_cols_limpos if row[col] == 1]

df['sintomas'] = df.apply(extrair_sintomas_linha_limpo, axis=1)

df_silver = df[[col_doenca, 'sintomas']].copy()
df_silver.head(10)

,diseases,sintomas
0,panic disorder,"[anxiety and nervousness, shortness of breath,..."
1,panic disorder,"[shortness of breath, depressive or psychotic ..."
2,panic disorder,"[anxiety and nervousness, depression, shortnes..."
3,panic disorder,"[anxiety and nervousness, depressive or psycho..."
4,panic disorder,"[anxiety and nervousness, depression, insomnia..."
5,panic disorder,"[shortness of breath, depressive or psychotic ..."
6,panic disorder,[anxiety and nervousness]
7,panic disorder,"[depressive or psychotic symptoms, insomnia, a..."
8,panic disorder,"[anxiety and nervousness, depressive or psycho..."
9,panic disorder,"[anxiety and nervousness, depression, shortnes..."


In [6]:
df_silver[col_doenca] = df_silver[col_doenca].str.lower().str.strip()

df_silver['sintomas'] = df_silver['sintomas'].apply(
    lambda lista: [s.replace('_', ' ').lower() for s in lista]
)

df_silver.head(10)

path_silver = "s3://silver/doencas_sintomas_limpo.csv"

df_silver.to_csv(path_silver, index=False, storage_options=minio_storage_options)

Agora no Gold, transformar em textos interpretáveis para o RAG

In [7]:
def gerar_texto(row):
    sintomas = ', '.join(row['sintomas'])
    return (
        f"{row['diseases'].capitalize()} is a condition that may present symptoms such as {sintomas}. "
        f"Individuals experiencing these symptoms may be associated with this condition. "
        f"This information is for educational purposes only and does not replace professional medical diagnosis."
    )

df_gold = df_silver.copy()
df_gold['texto'] = df_gold.apply(gerar_texto, axis=1)

df_gold.head(10)

path_gold = "s3://gold/textos_rag.csv"

df_gold.to_csv(path_gold, index=False, storage_options=minio_storage_options)

Possibilidade de enriquecimento dos textos para melhorar o desempenho do RAG

In [9]:
def generate_multiple_texts(row):
    symptoms = ', '.join(row['sintomas'])
    disease = row['diseases']

    return [
        f"{disease.capitalize()} may cause symptoms such as {symptoms}.",

        f"Patients experiencing {symptoms} may be associated with {disease}.",

        f"Common symptoms of {disease} include {symptoms}.",

        f"{disease.capitalize()} is often linked to symptoms like {symptoms}.",

        f"If a person presents with {symptoms}, {disease} could be a possible related condition."
    ]